# Analyse des cuissons — Four Verrier

Notebook de base : chargement depuis SQLite, graphe de profil température, analyse PID.

In [ ]:
import sqlite3
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DB_PATH = Path('..') / 'db' / 'kiln.db'
conn    = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f'Base : {DB_PATH.resolve()}')

## Sessions disponibles

In [ ]:
sessions = pd.read_sql("""
    SELECT s.id, s.log_type, s.program_name, s.start_datetime,
           s.firmware_version, s.kp, s.ki, s.kd, s.ema_alpha,
           s.num_steps, COUNT(d.id) AS nb_points
    FROM sessions s
    LEFT JOIN datapoints d ON d.session_id = s.id
    GROUP BY s.id
    ORDER BY s.start_datetime
""", conn)
sessions

## Profil de température — sélectionner une session

In [ ]:
SESSION_ID = 1  # ← changer ici

meta = dict(conn.execute('SELECT * FROM sessions WHERE id=?', (SESSION_ID,)).fetchone())
df   = pd.read_sql('SELECT * FROM datapoints WHERE session_id=? ORDER BY elapsed_sec',
                   conn, params=(SESSION_ID,))

print(f"Programme : {meta['program_name']}")
print(f"Début     : {meta['start_datetime']}")
print(f"Firmware  : {meta['firmware_version']}")
print(f"Kp={meta['kp']}  Ki={meta['ki']}  Kd={meta['kd']}  α={meta['ema_alpha']}")
if meta['steps_json']:
    steps = json.loads(meta['steps_json'])
    print('Étapes :')
    for i, s in enumerate(steps, 1):
        rate_str = f"{s['rate']}°C/min" if s['rate'] > 0 else 'libre'
        print(f"  {i}. {s['target']}°C  rampe={rate_str}  maintien={s['hold']//60} min")
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

elapsed_min = df['elapsed_sec'] / 60

# --- Température ---
ax = axes[0]
ax.plot(elapsed_min, df['target_temp'], '--', color='#FF6F00', lw=1.5,
        label='Consigne', zorder=2)
if 'raw_temp' in df.columns and df['raw_temp'].notna().any():
    ax.plot(elapsed_min, df['raw_temp'], color='#BDBDBD', lw=0.8,
            label='Brute (capteur)', zorder=1)
temp_col = 'filtered_temp' if ('filtered_temp' in df.columns and df['filtered_temp'].notna().any()) else 'current_temp'
ax.plot(elapsed_min, df[temp_col], color='#1B5E20', lw=2,
        label='Température filtrée', zorder=3)

# Zones de phase colorées
phase_colors = {'RAMP': '#E3F2FD', 'HOLD': '#FFF9C4',
                'STABILIZING': '#FCE4EC', 'BOOST': '#FBE9E7', 'IDLE': '#F5F5F5'}
prev_phase, prev_x = None, elapsed_min.iloc[0]
for _, row in df.iterrows():
    x = row['elapsed_sec'] / 60
    p = str(row['phase'])
    if p != prev_phase:
        if prev_phase and prev_phase in phase_colors:
            ax.axvspan(prev_x, x, color=phase_colors[prev_phase], alpha=0.35, zorder=0)
        prev_phase, prev_x = p, x
if prev_phase and prev_phase in phase_colors:
    ax.axvspan(prev_x, elapsed_min.iloc[-1], color=phase_colors[prev_phase], alpha=0.35, zorder=0)

ax.set_ylabel('Température (°C)')
ax.set_title(f"Profil — {meta['program_name']} — {meta['start_datetime']}")
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_locator(mticker.MultipleLocator(50))

# --- PWM SSR ---
ax2 = axes[1]
if 'ssr2_pwm' in df.columns and df['ssr2_pwm'].notna().any():
    ax2.fill_between(elapsed_min, df['ssr2_pwm'] / 1023 * 100,
                     color='#E64A19', alpha=0.7, label='Puissance SSR (%)')
    ax2.set_ylim(0, 110)
    ax2.set_ylabel('Puissance (%)')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
else:
    ax2.set_visible(False)

ax2.set_xlabel('Temps (min)')
plt.tight_layout()
plt.show()

## Analyse PID (logs maintenance uniquement)

In [ ]:
if meta['log_type'] != 'maintenance':
    print("Cette session n'est pas un log maintenance — cellule ignorée.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    ax = axes[0]
    ax.plot(elapsed_min, df['error'], color='#C62828', lw=1, label='Erreur (°C)')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel('Erreur (°C)')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(elapsed_min, df['p_term'], label='P', lw=1)
    ax.plot(elapsed_min, df['i_term'], label='I', lw=1)
    ax.plot(elapsed_min, df['d_term'], label='D', lw=1)
    ax.set_ylabel('Termes PID')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.fill_between(elapsed_min, df['ssr2_pwm'] / 1023 * 100,
                    color='#E64A19', alpha=0.7, label='Puissance (%)')
    ax.set_ylim(0, 110)
    ax.set_ylabel('Puissance (%)')
    ax.set_xlabel('Temps (min)')
    ax.legend(); ax.grid(True, alpha=0.3)

    axes[0].set_title(f'Analyse PID — {meta["program_name"]}')
    plt.tight_layout()
    plt.show()

## Comparaison multi-sessions

In [ ]:
# Comparer plusieurs sessions sur un même graphe
SESSION_IDS = [1]  # ← ajouter des IDs ici

fig, ax = plt.subplots(figsize=(14, 6))
for sid in SESSION_IDS:
    m  = dict(conn.execute('SELECT * FROM sessions WHERE id=?', (sid,)).fetchone())
    df2 = pd.read_sql('SELECT elapsed_sec, current_temp FROM datapoints WHERE session_id=? ORDER BY elapsed_sec',
                      conn, params=(sid,))
    label = f"{m['program_name']} ({m['start_datetime'][:10]})"
    ax.plot(df2['elapsed_sec'] / 60, df2['current_temp'], lw=1.5, label=label)

ax.set_xlabel('Temps (min)')
ax.set_ylabel('Température (°C)')
ax.set_title('Comparaison de sessions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()